In [7]:
import pandas as pd

# ============================================================
# DRUG-LEVEL GEPHI NETWORK
# ============================================================

signals = pd.read_csv("../data/signals/FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv")
atc = pd.read_csv("../data/raw/drug_atc_mapping.csv")

# Split PAIR into DRUG_A and DRUG_B
signals[['DRUG_A', 'DRUG_B']] = signals['PAIR'].str.split(' \\+ ', expand=True, n=1)

# Build ATC lookup: drug - RISK_CLASS
atc_lookup = dict(zip(atc['DRUG'], atc['RISK_CLASS']))

# Edges
edges = signals[['DRUG_A', 'DRUG_B', 'ROR', 'CI_LOWER', 'CI_UPPER', 'N_EXPOSED', 'PCT_SERIOUS']].copy()
edges.columns = ['Source', 'Target', 'ROR', 'CI_LOWER', 'CI_UPPER', 'N_EXPOSED', 'PCT_SERIOUS']
edges['Weight'] = edges['ROR']  # Gephi uses Weight for edge thickness
edges['Type'] = 'Undirected'

#ROR severity bins for edge coloring in Gephi
edges['severity'] = pd.cut(edges['ROR'], 
                           bins=[0, 3, 5, 10, float('inf')], 
                           labels=['mild', 'moderate', 'strong', 'severe'])

# Nodes
# Count how many signals each drug appears in
drug_counts = pd.concat([
    signals['DRUG_A'].rename('DRUG'),
    signals['DRUG_B'].rename('DRUG')
]).value_counts().reset_index()
drug_counts.columns = ['Id', 'signal_count']

# Map ATC class
drug_counts['drug_class'] = drug_counts['Id'].map(atc_lookup).fillna('Unclassified')

# Color map (same as class-level network)
COLOR_MAP = {
    'Immunosuppressant': (230, 57, 70),
    'Anti-inflammatory (NSAID)': (231, 111, 81),
    'Corticosteroid': (244, 162, 97),
    'Analgesic/Opioid': (38, 70, 83),
    'Antineoplastic': (42, 157, 143),
    'Antithrombotic': (69, 123, 157),
    'Alimentary/Metabolism': (138, 177, 125),
    'Antibacterial': (106, 153, 78),
    'Anti-infective (Other)': (56, 102, 65),
    'Antiviral': (77, 144, 142),
    'Respiratory (Other)': (144, 190, 109),
    'Cardiovascular (Other)': (87, 117, 144),
    'RAAS Inhibitor': (67, 97, 160),
    'Lipid Modifying': (123, 151, 212),
    'Psycholeptic': (155, 93, 229),
    'Psychoanaleptic': (177, 133, 219),
    'Antiepileptic': (201, 173, 229),
    'Nervous System (Other)': (212, 191, 239),
    'Dermatological': (188, 108, 37),
    'Musculoskeletal (Other)': (221, 161, 94),
    'Antidiabetic': (96, 108, 56),
    'Blood (Other)': (61, 90, 128),
    'Anesthetic': (119, 141, 169),
    'Antiparasitic': (163, 177, 138),
    'Various': (153, 153, 153),
    'Unclassified': (204, 204, 204),
}

drug_counts['r'] = drug_counts['drug_class'].map(lambda x: COLOR_MAP.get(x, (204,204,204))[0])
drug_counts['g'] = drug_counts['drug_class'].map(lambda x: COLOR_MAP.get(x, (204,204,204))[1])
drug_counts['b'] = drug_counts['drug_class'].map(lambda x: COLOR_MAP.get(x, (204,204,204))[2])
drug_counts['Label'] = drug_counts['Id']

# --- SAVE FULL NETWORK ---
drug_counts.to_csv("../network/gephi_nodes.csv", index=False)
edges.to_csv("../network/gephi_edges.csv", index=False)
print(f"Full network: {len(drug_counts)} nodes, {len(edges)} edges")

# --- TOP 100 BY N_EXPOSED ---
top100_edges = edges.nlargest(100, 'N_EXPOSED').copy()
top100_drugs = set(top100_edges['Source']) | set(top100_edges['Target'])
top100_nodes = drug_counts[drug_counts['Id'].isin(top100_drugs)].copy()

top100_nodes.to_csv("../network/gephi_nodes_top100.csv", index=False)
top100_edges.to_csv("../network/gephi_edges_top100.csv", index=False)
print(f"Top 100 network: {len(top100_nodes)} nodes, {len(top100_edges)} edges")

# --- TOP 50 BY N_EXPOSED ---
top50_edges = edges.nlargest(50, 'N_EXPOSED').copy()
top50_drugs = set(top50_edges['Source']) | set(top50_edges['Target'])
top50_nodes = drug_counts[drug_counts['Id'].isin(top50_drugs)].copy()

top50_nodes.to_csv("../network/gephi_nodes_top50.csv", index=False)
top50_edges.to_csv("../network/gephi_edges_top50.csv", index=False)
print(f"Top 50 network: {len(top50_nodes)} nodes, {len(top50_edges)} edges")

# --- SUMMARY ---
print("\n--- Drug class distribution (top 100 network) ---")
print(top100_nodes['drug_class'].value_counts().to_string())
print(f"\n--- ROR range (top 100) ---")
print(f"Min: {top100_edges['ROR'].min():.1f}, Max: {top100_edges['ROR'].max():.1f}, Median: {top100_edges['ROR'].median():.1f}")
print(f"\n--- Severity distribution (top 100) ---")
print(top100_edges['severity'].value_counts().to_string())

Full network: 682 nodes, 5501 edges
Top 100 network: 67 nodes, 100 edges
Top 50 network: 38 nodes, 50 edges

--- Drug class distribution (top 100 network) ---
drug_class
Alimentary/Metabolism        9
Analgesic/Opioid             7
Antineoplastic               7
Psycholeptic                 5
Corticosteroid               4
Immunosuppressant            4
Antibacterial                4
Antithrombotic               3
Beta Blocker                 2
Antiviral                    2
Narrow Therapeutic Index     2
Anti-asthmatic               2
Antidiabetic                 2
Psychoanaleptic              2
Lipid Modifying              2
Cardiovascular (Other)       2
RAAS Inhibitor               2
Anti-inflammatory (NSAID)    1
Musculoskeletal (Other)      1
Various                      1
Antiparasitic                1
Blood (Other)                1
Hormonal                     1

--- ROR range (top 100) ---
Min: 2.0, Max: 4.8, Median: 2.4

--- Severity distribution (top 100) ---
severity
mild  

In [8]:
# Fix: add missing classes to color map and regenerate
FULL_COLOR_MAP = {
    'Immunosuppressant': (230, 57, 70),
    'Anti-inflammatory (NSAID)': (231, 111, 81),
    'Corticosteroid': (244, 162, 97),
    'Analgesic/Opioid': (38, 70, 83),
    'Antineoplastic': (42, 157, 143),
    'Antithrombotic': (69, 123, 157),
    'Alimentary/Metabolism': (138, 177, 125),
    'Antibacterial': (106, 153, 78),
    'Anti-infective (Other)': (56, 102, 65),
    'Antiviral': (77, 144, 142),
    'Respiratory (Other)': (144, 190, 109),
    'Anti-asthmatic': (181, 201, 154),
    'Cardiovascular (Other)': (87, 117, 144),
    'RAAS Inhibitor': (67, 97, 160),
    'Beta Blocker': (92, 127, 191),
    'Lipid Modifying': (123, 151, 212),
    'Psycholeptic': (155, 93, 229),
    'Psychoanaleptic': (177, 133, 219),
    'Antiepileptic': (201, 173, 229),
    'Nervous System (Other)': (212, 191, 239),
    'Dermatological': (188, 108, 37),
    'Musculoskeletal (Other)': (221, 161, 94),
    'Antidiabetic': (96, 108, 56),
    'Genitourinary': (204, 213, 174),
    'Blood (Other)': (61, 90, 128),
    'Anesthetic': (119, 141, 169),
    'Antiparasitic': (163, 177, 138),
    'Narrow Therapeutic Index': (178, 34, 34),
    'Hormonal': (199, 168, 125),
    'Various': (153, 153, 153),
    'Unclassified': (204, 204, 204),
}

import pandas as pd

for suffix in ['', '_top50', '_top100']:
    nodes = pd.read_csv(f"gephi_nodes{suffix}.csv")
    nodes['r'] = nodes['drug_class'].map(lambda x: FULL_COLOR_MAP.get(x, (204,204,204))[0])
    nodes['g'] = nodes['drug_class'].map(lambda x: FULL_COLOR_MAP.get(x, (204,204,204))[1])
    nodes['b'] = nodes['drug_class'].map(lambda x: FULL_COLOR_MAP.get(x, (204,204,204))[2])
    nodes.to_csv(f"gephi_nodes{suffix}.csv", index=False)

print("All node files updated with complete color map")

All node files updated with complete color map
